![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🗺️ Nova Scotia Health Management Zones — Data Cleaning Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-NS_Health_Authority-1A6B5A?style=flat-square)

> **Purpose:** Build the `DimHealthZone` dimension table — the geographic backbone that connects population, chronic disease, and health outcome data across all Power BI pages.

This notebook cleans the NS community cluster population dataset, which maps the hierarchy of health management zones → community health networks → community clusters. It is a dimension table, not a fact table — it provides the descriptive context for filtering and grouping in Power BI.

---

## 📦 Source Data

| File | Rows | Coverage |
|------|------|----------|
| `Nova_Scotia_Community_Clusters_Census_Population_20260424.csv` | 108 | 4 health zones, 2011 + 2016 census years |

**Geographic hierarchy:**
```
Province (Nova Scotia)
  └── Health Zone (1–4)
        └── Community Health Network
              └── Community Cluster
```

---

## 🔧 Cleaning Steps Overview

| Step | Action |
|------|--------|
| 1 | Imports & load |
| 2 | Normalize column names (strip whitespace) |
| 3 | Split `Zone` string into code + name |
| 4 | Clean numeric columns (remove comma formatting) |
| 5 | Cast ID columns to integer |
| 6 | Select, rename, and order final columns |
| 7 | Sort and validate |
| 8 | Export |

---


## Step 1 — Import & Load Raw Data

The source file is already in a relatively clean structure — the main work is column name normalization, zone string splitting, and type casting.


In [ ]:
import pandas as pd

df = pd.read_csv(
    './data/Nova_Scotia_Community_Clusters_Census_Population_20260424.csv'
)

print(f"Raw shape: {df.shape}")
print(f"Columns  : {df.columns.tolist()}")
print()
print(df.head(3))


## Step 2 — Normalize Column Names

Strip leading/trailing whitespace from all column names. StatCan and NSHA files sometimes include invisible whitespace that causes `KeyError` when accessing columns by name.


In [ ]:
df.columns = df.columns.str.strip()
print("Columns after strip:", df.columns.tolist())


## Step 3 — Split Health Zone String

The raw `Zone` column contains a combined code and name e.g. `'ZONE 1: WESTERN'`.

We split on `':'` to create two separate columns that match the `DimHealthZone` schema:
- `Health Zone` → `'ZONE 1'` (zone code — join key)
- `Health Zone Name` → `'WESTERN'` (display label)


In [ ]:
df[['Health Zone', 'Health Zone Name']] = (
    df['Zone']
    .str.split(':', expand=True)
)

df['Health Zone']      = df['Health Zone'].str.strip()
df['Health Zone Name'] = df['Health Zone Name'].str.strip()

print("Zone split sample:")
print(df[['Zone', 'Health Zone', 'Health Zone Name']].drop_duplicates().head(8))


## Step 4 — Clean Numeric Columns

Population columns may contain comma-formatted numbers (e.g. `'26,453'`). We strip commas and cast to float.


In [ ]:
numeric_cols = [
    'Population Un-Weighted',
    'Population Weight',
    'Population After Weighing'
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

print("Numeric columns after cleaning:")
print(df[numeric_cols].describe())


## Step 5 — Cast ID Columns to Integer

Zone, Network, and Cluster IDs are stored as floats in the raw file (a common pandas artefact when a column has mixed types or was read from Excel). We cast them to integer for clean join keys.


In [ ]:
df['ZoneID']     = df['ZoneID'].astype(int)
df['NetworkID']  = df['NetworkID'].astype(int)
df['ClusterID']  = df['ClusterID'].astype(int)

print("ID columns cast to int:")
print(df[['ZoneID', 'NetworkID', 'ClusterID']].dtypes)


## Step 6 — Select, Rename & Order Final Columns

Rename `ZoneID` → `Health ZoneID` to match the schema convention used across all other datasets in the project. Drop the temporary `Zone` string column.


In [ ]:
df = df.rename(columns={'ZoneID': 'Health ZoneID'})

community_cluster_population = df[[
    'Health ZoneID',
    'Census Year',
    'Health Zone',
    'Health Zone Name',
    'ClusterID',
    'Community Cluster',
    'Community Health Network',
    'NetworkID',
    'Population Un-Weighted',
    'Population Weight',
    'Population After Weighing'
]]


## Step 7 — Sort & Validate


In [ ]:
community_cluster_population = community_cluster_population.sort_values(
    ['Census Year', 'Health ZoneID', 'NetworkID', 'ClusterID']
)

print(community_cluster_population.info())
print()
print(community_cluster_population.head())


## Step 8 — Export


In [ ]:
import os
os.makedirs("./clean", exist_ok=True)

community_cluster_population.to_csv(
    './clean/community_cluster_population.csv',
    index=False
)

print("✅ Saved: ./clean/community_cluster_population.csv")
print(f"✅ Rows : {len(community_cluster_population)}")


## ✅ Output Summary

| Output | Location | Rows | Columns |
|--------|----------|------|---------|
| Clean CSV | `./clean/community_cluster_population.csv` | 108 | 11 |

**Final schema:**
`Health ZoneID` \| `Census Year` \| `Health Zone` \| `Health Zone Name` \| `ClusterID` \| `Community Cluster` \| `Community Health Network` \| `NetworkID` \| `Population Un-Weighted` \| `Population Weight` \| `Population After Weighing`

**Role in the star schema:** This is `DimHealthZone` — it joins to `fact_chronic_disease` on `Health Zone` to enable zone-level filtering and the NS choropleth map in Power BI.


---

*Nova Scotia Health & Population Analytics · NS Health Authority*
